# Lesson 5: Personal Chef Project

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Any:
    """Useful for searching the web for recipes and ingredients.
    Pass natural language search queries like 'easy eggs spinach cheese recipes'.
    Do NOT use search operators like 'site:' alone.
    """
    try:
        results = tavily_client.search(query=query, max_results=3)
        return results
    except Exception as e:
        # Returning the error string lets the agent read it, self-correct, and try another query!
        return f"Search error for query '{query}': {str(e)}. Please retry with different keywords."

In [11]:
system_prompt = """
    You are a personal chef that is very good at making the most of very little. The user will give you a list of ingredients or an image of their fridge or pantry (or, perhaps, both) and, based on that information, use the web search tool to find recipes that can be made with those ingredients. You will then return a list of recipes, along with the ingredients needed for each recipe and the steps to make it. Do not make up recipes or ingredients. Do not return recipes that contain ingredients that the person does not have. If you cannot find any recipes that can be made with the given ingredients, you will return a message saying so. You will also provide a list of substitutions for any ingredients that are not available, if possible.
"""

In [18]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 1. Initialize Multimodal Gemini
gemini = init_chat_model(
    model="models/gemini-flash-latest", 
    model_provider="google_genai"
)

# 2. Checkpointer for conversation memory
checkpointer = InMemorySaver()

# 3. Create the Chef Agent
chef_agent = create_agent(
    model=gemini,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [30]:
gemini_lite = init_chat_model(
    model="models/gemini-3.5-flash-lite", 
    model_provider="google_genai"
)

chef_agent = create_agent(
    model=gemini_lite,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer  # <--- Re-uses the existing memory!
)

In [19]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [20]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [22]:
from langchain.messages import HumanMessage

# 2. Build the multimodal message
question = HumanMessage(content=[
    {"type": "text", "text": "Here is a photo of what I have in my fridge. What recipes can I make tonight?"},
    {
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
    }
])

# 3. Set a thread_id for conversational memory
config = {"configurable": {"thread_id": "fridge_convo_1"}}

# 4. Invoke the agent
response = chef_agent.invoke({"messages": [question]}, config=config)

# 5. Print the chef's recommendations
ans = response['messages'][-1].content
print(ans[0]['text'] if isinstance(ans, list) else ans)

Based on the ingredients visible in your fridge, here is what you have available:

* **Proteins & Dairy:** Fresh eggs (top door shelf), canned fish/tuna (*Coqueiro*), assorted cheeses (white cheese, yellow cheese, quark tubs), sliced deli meat/ham (clear container), skim milk (*Parmalat*).
* **Produce:** Fresh broccoli, green onions/herbs (crisper drawer).
* **Pantry & Condiments:** Cooking oil, vinegar (*Rosani*), mustard, ketchup, hot pepper sauce (*Molho de Pimenta*), salt/seasoning.

Here are a few quick, complete recipes you can make tonight using only these ingredients:

---

### 1. Skillet Ham, Broccoli & Cheese Frittata

**Ingredients needed:**
* 3–4 eggs
* 1–2 cups fresh broccoli florets, chopped small
* 2–3 slices deli ham, chopped
* ¼ cup grated or crumbled cheese (your yellow cheese or white cheese)
* 2 tbsp milk
* 1 tbsp cooking oil
* Chopped green onions (optional, for flavor)
* Salt and black pepper to taste

**Steps:**
1. **Sauté the vegetables and ham:** Heat the oil i

In [31]:
follow_up = HumanMessage(content="I noticed there is a bottle of Garibaldi wine on the bottom door shelf. Which of those 3 recipes would pair best with it and why?")

response_2 = chef_agent.invoke({"messages": [follow_up]}, config=config)

ans_2 = response_2['messages'][-1].content
print(ans_2[0]['text'] if isinstance(ans_2, list) else ans_2)

### 1. Stovetop Alternative: Broccoli, Ham & Cheese Soft Scramble (No Oven Needed!)

If you want to skip the frittata format altogether, you can make a fast and fluffy stovetop **Broccoli, Ham & Cheese Soft Scramble** in under 10 minutes:

* **How to make it:** 
  1. In a skillet, sauté the chopped broccoli with a splash of water and oil until tender, then toss in your chopped deli ham to warm through.
  2. Beat your eggs in a bowl with a splash of milk, salt, and pepper. 
  3. Lower the heat under the skillet, pour the eggs directly over the broccoli and ham, and gently push the eggs around with a spatula continuously over medium-low heat until soft, creamy curds form.
  4. Fold in your cheese right at the end so it melts into the eggs, and serve immediately!

---

### 2. Wine Pairing: Which recipe pairs best with the Garibaldi red wine?

The **Skillet Ham, Broccoli & Cheese Frittata** (or the Stovetop Scramble above) pairs best with the **Garibaldi red wine**. 

**Why it works:**
* *